# Principal Component Analysis (PCA)

[Principal Component Analysis (PCA)](https://en.wikipedia.org/wiki/Principal_component_analysis) is a statistical procedure that converts a set of observations of possibly correlated variables into a set of values of linearly uncorrelated variables. Each of the principal components is chosen in such a way that it would describe most of the still available variance and all these principal components are orthogonal to each other.

For example, in the image below, there is a collection of data points that are made up of two features, Feature 1 and Feature 2. The red arrow is parallel to the first principal component, which is also the direction of the highest variance. The green arrow is parallel to the second principal component, and it is the direction of highest variance orthogonal to the first one. 

![PCA eigenvectors](img/pca_eigen.png)

After calculating the principal components, we can use them to transform the data into a new space. The new space is a subspace of the original space, where the first principal component is the direction of the highest variance. The second principal component is the direction of the highest variance orthogonal to the first one. The third principal component is the direction of the highest variance orthogonal to the first two, and so on. In this new space we can drop a dimension of low variance, and project the data points to the remaining dimensions. By only dropping dimensions of low variance, we can reduce the dimensionality of the data, while still retaining the majority of the information.

![PCA project](img/pca-projection.png)

Effectively, we can use the projection matrix of eigenvectors computed by PCA to represent high dimensional data in a low dimensional space, and we can use the eigenvectors to transform the data back to the original space. During this transformation process some information is lost, so the reconstructed data are not exactly the same as the original data, but they are close.

Let's now dive into how PCA works.

- **Step 1:** Load a set of face images and convert each one to a vector. A grayscale image of $H \times W$ pixels becomes a 1-D array of $d = H \cdot W$ numbers, its rows laid end to end. Stacking the $n$ vectors as rows gives an $n \times d$ data matrix $\mathbf{X}$.
- **Step 2:** Compute the mean of the vectors (the *average face*) and subtract it from every vector, so that the data are centered on zero.
- **Step 3:** Compute the covariance matrix $\mathbf{C}$, a $d \times d$ matrix. Its entry $C_{ij}$ is the covariance between pixel $i$ and pixel $j$ across all the images: positive when the two pixels tend to be bright together, negative when one tends to be bright while the other is dark. The diagonal entry $C_{ii}$ is the variance of pixel $i$.
- **Step 4:** Compute the eigenvectors and eigenvalues of $\mathbf{C}$. Each eigenvector is a direction in the $d$-dimensional space (a unit vector), and its eigenvalue is the variance of the data along that direction. The eigenvector with the largest eigenvalue is the first principal component.
- **Step 5:** Project a vector onto the top $k$ eigenvectors: subtract the mean, then take the dot product with each eigenvector. The result is $k$ numbers.
- **Step 6:** Reconstruct (back-project) the vector from those $k$ numbers: multiply each number by its eigenvector, add them up, and add the mean back.

We will use a dataset of 337 face images to demonstrate how PCA works in practice. (The face recognition example that used to end this notebook now lives in the notebook for the next lecture, nearest-neighbor recognition.)

### Load the dataset

In [ ]:
# Necessary imports
import cv2 as cv
import numpy as np
from matplotlib import pyplot as plt
import os

# Step 1: Load the images and convert them to vectors
# sorted(): os.listdir's order depends on the file system; without it,
# face 40 below would be a different face on different machines
filenames = ['data/faces/' + filename for filename in sorted(os.listdir('data/faces'))]

face_vectors = []
for filename in filenames:
    img = cv.imread(filename, cv.IMREAD_GRAYSCALE)
    img = img[25:85, 25:75]  # Crop the face

    img_vector = img.flatten()  # 60x50 image -> 1-D array of 3000 values
    face_vectors.append(img_vector)  # Add the vector to the list

# Convert the face vectors to a numpy array
face_vectors = np.array(face_vectors)
face_vectors = face_vectors.astype(np.float32)

# Normalize each face to mean 0 and std 1 (brightness and contrast no longer matter)
for i in range(face_vectors.shape[0]):
    face_vectors[i, :] = (
        face_vectors[i, :] - np.mean(face_vectors[i, :])) / np.std(face_vectors[i, :])

print('Data matrix:', face_vectors.shape)   # (number of faces, number of pixels)

# Visualize a few sample faces
plt.figure(figsize=(10, 10))
for i in range(9):
    plt.subplot(3, 3, i+1)
    plt.imshow(face_vectors[i].reshape(60, 50), cmap='gray')
    plt.axis('off')


### Apply PCA to compute the Eigenfaces (eigenvectors of faces)

In [ ]:
# Step 2: Compute the average face vector
average_face_vector = np.mean(face_vectors, axis=0)

# Center each face vector by subtracting the average face vector
face_vectors_centered = face_vectors - average_face_vector

# Step 3: Compute the covariance matrix
# covariance_matrix = np.cov(face_vectors_centered, rowvar=False)
covariance_matrix = np.dot(
    face_vectors_centered.T, face_vectors_centered) / (face_vectors_centered.shape[0] - 1)

# Step 4: Compute the eigenvectors and eigenvalues
eigenvalues, eigenvectors = np.linalg.eigh(covariance_matrix)

# Note: The eigenvectors are the COLUMNS of `eigenvectors`, and eigh returns the eigenvalues in ascending order
print('Covariance matrix:', covariance_matrix.shape)
print('Largest three eigenvalues:', eigenvalues[-3:][::-1])

### Visualize Eigenfaces

We have now computed the eigenvectors and eigenvalues of the covariance matrix. We can now use the eigenvectors to project the data onto the principal components. Before we do so, let's first visualize the top 10 (most significant) eigenvectors to see what they look like. Those will resemble faces. Each one of the eigenvectors captures some property of the face. If, on the other hand, we visualize the bottom 10 (least significant) eigenvectors, they will mostly appear as noise. To visualize the eigenvectors we reshape them back to 2D images. Those images are also known as **eigenfaces**.

In [ ]:
# Sort the eigenvectors and eigenvalues in descending order of eigenvalues
idx = eigenvalues.argsort()[::-1]
eigenvalues = eigenvalues[idx]
eigenvectors = eigenvectors[:, idx]

# Visualize the top 10 eigenvectors
plt.figure(figsize=(10, 5))
for i in range(10):
    plt.subplot(2, 5, i+1), plt.imshow(eigenvectors[:, i].reshape(60, 50), cmap='gray')
    plt.title("Eigenface " + str(i + 1)), plt.xticks([]), plt.yticks([])
plt.show()

In [ ]:
# Visualize the bottom 10 eigenvectors
plt.figure(figsize = (10,5))
for i in range(10):
    index = eigenvectors.shape[1] - (10 - i)
    plt.subplot(2,5,i+1), plt.imshow(eigenvectors[:, index].reshape(60,50), cmap='gray')
    plt.title("Eigenvector " + str(index)), plt.xticks([]), plt.yticks([])
plt.show()

The bottom eigenvectors look like noise for a reason. We have only $n = 337$ faces, and after subtracting the mean they span at most $n - 1 = 336$ directions of the 3000-dimensional space. Along every other direction the faces do not vary at all, so the remaining 2,664 eigenvalues are zero. Their eigenvectors are an arbitrary set of directions that `eigh` had to return, and rounding in `float32` even makes a few of the zero eigenvalues slightly negative.

In [ ]:
n = face_vectors.shape[0]
print('Number of faces:', n)
print('Eigenvalues 335 to 338:', eigenvalues[334:338])
print('Eigenvalues greater than 1e-3:', np.sum(eigenvalues > 1e-3))
print('Bottom 10 eigenvalues:', eigenvalues[-10:])

### How Much Variance Does Each Eigenface Keep?

The eigenvalue of an eigenface is the variance of the faces along it, and the eigenvalues add up to the total variance of the data. So the share of the total that the top $k$ eigenvalues hold is the share of the variation that $k$ numbers per face can keep. This curve is the usual way to choose $k$.

In [ ]:
explained = eigenvalues / np.sum(eigenvalues)
cumulative = np.cumsum(explained)

plt.figure(figsize=(12, 4))
plt.subplot(121)
plt.plot(np.arange(1, 51), eigenvalues[:50], 'o-', markersize=3)
plt.xlabel('eigenface number'), plt.ylabel('eigenvalue (variance)'), plt.title('Top 50 eigenvalues')
plt.subplot(122)
plt.plot(np.arange(1, n), cumulative[:n - 1])
plt.xlabel('number of eigenfaces k'), plt.ylabel('share of the variance kept'), plt.title('Cumulative variance')
plt.grid(True)
plt.show()

for k in (1, 10, 50, 100, 200, 336):
    print(f'k = {k:3d}: {cumulative[k - 1]:.1%} of the variance')

### What Each Eigenface Changes

An eigenface is not a face. It is a *direction of change*: a pattern of pixels to add to, or subtract from, the average face. Its values are positive and negative, and `plt.imshow` stretches them to black and white, which is why eigenfaces look like ghostly faces. Its sign is arbitrary too: `eigh` could just as well have returned the negative of each one, which would look like a photographic negative.

To see what an eigenface does, start from the average face and walk along it, two standard deviations each way. The standard deviation along eigenface $i$ is $\sqrt{\lambda_i}$.

In [ ]:
plt.figure(figsize=(9, 10))
for row in range(3):
    sd = np.sqrt(eigenvalues[row])
    for col, steps in enumerate((-2, 0, 2)):
        face = average_face_vector + steps * sd * eigenvectors[:, row]
        plt.subplot(3, 3, 3 * row + col + 1)
        plt.imshow(face.reshape(60, 50), cmap='gray', vmin=-3, vmax=3)   # one gray scale for all nine
        plt.title(f'average {steps:+d} sd x eigenface {row + 1}' if steps else 'average face')
        plt.xticks([]), plt.yticks([])
plt.show()

### Project faces to lower dimension

Let's now see how we can reduce the dimensionality of a face image by projecting it onto a few principal components. We will start by projecting the face to only the **first principal component**.

In [ ]:
face_vector = face_vectors[40, :] # Select one face (number 40)

# Center the face vector by subtracting the average face vector
face_vector_centered = face_vector - average_face_vector

num_of_eigenvectors = 1     # Number of eigenvectors to use

# Expand dimensions of the face vector from (3000,) to (1,3000) to perform matrix multiplication
face_vector_centered = np.expand_dims(face_vector_centered, axis=0)

# Project the face vector onto the eigenvectors to get the feature vector
# (1,3000) x (3000,1) = (1,1)
projected_face = face_vector_centered.dot(eigenvectors[:, :num_of_eigenvectors])

print('Projected face: ', projected_face)

### Reconstruct (backproject) faces to original dimension
As we can see, the result of projecting a face onto the first principal component is a single number. That is expected: the product of matrices of shapes (1, 3000) and (3000, 1) is a (1, 1) matrix, one number. Effectively, we are approximating the face with a single number. Let's now try to reconstruct the face by back-projecting it to its original shape. To do so, we need to multiply the projected data by the first eigenvector and add the mean.

In [ ]:
# Reconstruct the face vector from the feature vector
# (1, 1) x (1,3000) = (1,3000)
reconstructed_face_vector = projected_face.dot(eigenvectors[:, :num_of_eigenvectors].T) + average_face_vector

# Visualize the reconstructed face
face = face_vector.reshape(60, 50)
reconstructed_face = reconstructed_face_vector.reshape(60, 50)

plt.figure(figsize=(10, 5))
plt.subplot(121)
plt.imshow(face, cmap='gray'), plt.title('Original image')
plt.subplot(122)
plt.imshow(reconstructed_face, cmap='gray'), plt.title('Reconstructed image')
plt.show()

### A Face Is a Recipe

With 10 eigenvectors the projection is 10 numbers. Each number says how much of one eigenface to add to the average face, so the reconstruction is a weighted sum:

$$\text{face} \approx \text{average face} + c_1 \cdot \text{eigenface}_1 + c_2 \cdot \text{eigenface}_2 + \dots + c_{10} \cdot \text{eigenface}_{10}$$

This is the same kind of linear combination as the average of five faces in the lecture, with the weights chosen by projection instead of all set to 1/5.

In [ ]:
num_of_eigenvectors = 10     # Number of eigenvectors to use

projected_face = face_vector_centered.dot(eigenvectors[:, :num_of_eigenvectors])
print('The 10 numbers:', np.round(projected_face, 1))

# The reconstruction, written out as the weighted sum
recipe = average_face_vector.copy()
for i in range(num_of_eigenvectors):
    recipe += projected_face[0, i] * eigenvectors[:, i]

# ... is exactly what the matrix product computes
reconstructed_face_vector = projected_face.dot(eigenvectors[:, :num_of_eigenvectors].T) + average_face_vector
print('Same as the matrix product:', np.allclose(recipe, reconstructed_face_vector[0], atol=1e-4))

### Approximating the Face With More and More Numbers

The same projection and reconstruction, for several values of $k$. Under each picture: the share of the dataset's variance that $k$ eigenfaces keep, and the reconstruction error of this face, the length of (face minus reconstruction) relative to the length of the face.

In [ ]:
def reconstruct(face, k):
    """Project a face onto the top k eigenvectors and reconstruct it."""
    centered = face - average_face_vector
    weights = centered.dot(eigenvectors[:, :k])
    return average_face_vector + eigenvectors[:, :k].dot(weights)

ks = [0, 1, 2, 3, 5, 10, 50, 100, 200]
plt.figure(figsize=(18, 3.2))
plt.subplot(1, len(ks) + 1, 1)
plt.imshow(face_vector.reshape(60, 50), cmap='gray', vmin=-3, vmax=3), plt.title('original', fontsize=10), plt.axis('off')
for j, k in enumerate(ks):
    rec = reconstruct(face_vector, k)
    error = np.linalg.norm(face_vector - rec) / np.linalg.norm(face_vector)
    kept = cumulative[k - 1] if k > 0 else 0
    plt.subplot(1, len(ks) + 1, j + 2)
    plt.imshow(rec.reshape(60, 50), cmap='gray', vmin=-3, vmax=3)
    plt.title(f'k = {k}\n{kept:.0%} of var.\nerror {error:.2f}', fontsize=10), plt.axis('off')
plt.show()

With 0 numbers we get the average face, the best guess when we know nothing about the face. With 10 numbers the reconstruction already looks like this person, and with 100 or 200 it is hard to tell from the original. We have reduced the face from 3000 numbers to 100 or 200 and kept most of the information.

### PCA Is Not Resizing

A smaller image is also a face described with fewer numbers. Which is better with the same budget: $k$ eigenface weights, or a $k$-pixel thumbnail? To be fair to the thumbnail, the eigenfaces here are computed *without* the face being reconstructed, so PCA has never seen it.

In [ ]:
others = np.delete(face_vectors, 40, axis=0)                     # every face except number 40
mean_others = np.mean(others, axis=0)
eigvals_o, eigvecs_o = np.linalg.eigh(np.cov(others - mean_others, rowvar=False))
eigvecs_o = eigvecs_o[:, ::-1]

for k, (h, w) in [(12, (4, 3)), (30, (6, 5)), (120, (12, 10))]:
    weights = (face_vector - mean_others).dot(eigvecs_o[:, :k])
    pca_rec = mean_others + eigvecs_o[:, :k].dot(weights)
    small = cv.resize(face_vector.reshape(60, 50), (w, h), interpolation=cv.INTER_AREA)
    resized = cv.resize(small, (50, 60), interpolation=cv.INTER_LINEAR).flatten()
    err_pca = np.linalg.norm(face_vector - pca_rec) / np.linalg.norm(face_vector)
    err_small = np.linalg.norm(face_vector - resized) / np.linalg.norm(face_vector)
    print(f'{k:3d} numbers: PCA error {err_pca:.2f}, {h}x{w} thumbnail error {err_small:.2f}')

PCA wins at every budget. A thumbnail keeps the same few pixels for every image, whatever the image shows. PCA chooses its few numbers by learning from 336 other faces which patterns faces actually vary in. The price is that the eigenfaces only suit faces: a thumbnail of a tree is still a tree, while the eigenface reconstruction of a tree is a face.

### Implementations of PCA, Projection, and Reconstruction as functions

In [ ]:
# A function that computes the PCA of a set of vectors
def computePCA(data_vectors, average_vector, n_components):
    # n centered vectors span at most n - 1 directions (and at most d),
    # so only that many eigenvalues can be non-zero
    max_components = min(data_vectors.shape[0] - 1, data_vectors.shape[1])
    if n_components > max_components:
        print("Only", max_components, "components have non-zero variance")
        n_components = max_components

    # Center each vector around zero by subtracting the average vector
    data_vectors_centered = data_vectors - average_vector

    # Compute the covariance matrix
    covariance_matrix = np.cov(data_vectors_centered, rowvar=False)

    # Compute the eigenvectors and eigenvalues
    eigenvalues, eigenvectors = np.linalg.eigh(covariance_matrix)

    # Sort the eigenvectors and eigenvalues in descending order of eigenvalues
    idx = eigenvalues.argsort()[::-1]
    eigenvalues = eigenvalues[idx]
    eigenvectors = eigenvectors[:, idx]

    return eigenvectors[:, :n_components]

# A function that projects a set of vectors onto a set of eigenvectors
def pca_project(data_vectors, eigenvectors, average_vector):
    # If the data vectors is a vector, convert it to a matrix
    if len(data_vectors.shape) == 1:
        data_vectors = np.expand_dims(data_vectors, axis=0)

    data_vectors_centered = data_vectors - average_vector
    projected_data = data_vectors_centered.dot(eigenvectors)
    return projected_data

# A function that reconstructs a set of vectors from a set of eigenvectors
def pca_reconstruct(projected_data, eigenvectors, average_vector):
    # If the projected data is a vector, convert it to a matrix
    if len(projected_data.shape) == 1:
        projected_data = np.expand_dims(projected_data, axis=0)

    # Reconstruct the data
    reconstructed_data = projected_data.dot(eigenvectors.T) + average_vector
    return reconstructed_data


### The Same Thing in scikit-learn

In practice you would not write PCA yourself. `sklearn.decomposition.PCA` centers the data, finds the components (with an SVD, which gives the same directions as the eigenvectors of the covariance matrix) and sorts them for you. Its `components_` are our eigenvectors as rows, possibly with the opposite sign, since the sign of an eigenvector is arbitrary.

In [ ]:
from sklearn.decomposition import PCA

# svd_solver='full' asks for the exact answer; for a small number of components the default
# switches to a faster randomized method whose components are only approximately the same
pca = PCA(n_components=10, svd_solver='full').fit(face_vectors)
print('Explained variance ratio of the top 3:', np.round(pca.explained_variance_ratio_[:3], 4))
print('Ours:                                 ', np.round(explained[:3], 4))

# Each sklearn component equals one of our eigenvectors, up to sign: |dot product| = 1
print('|dot products|:', np.round(np.abs(np.sum(pca.components_ * eigenvectors[:, :10].T, axis=1)), 4))

# transform = our projection, inverse_transform = our reconstruction (up to the signs)
weights = pca.transform(face_vectors[40:41])
print('Same reconstruction:', np.allclose(pca.inverse_transform(weights)[0], reconstruct(face_vector, 10), atol=1e-3))

## Faces vs. Non-Faces: Reconstruction Error

Eigenfaces are good at reconstructing faces, because that is what they were learned from. A window that is not a face reconstructs badly, since no weighted sum of eigenfaces looks like a tree or a wall. So the reconstruction error is a face detector's score: the lower it is, the more face-like the window.

`data/faces1000.npy` holds 1000 face windows and `data/nonfaces1000.npy` 1000 windows of other things, each $31 \times 25$ pixels. We learn eigenfaces from 500 of the faces and score the other 500 faces and 500 non-faces.

In [ ]:
faces = np.load('data/faces1000.npy').astype(np.float32)        # (1000, 31, 25)
nonfaces = np.load('data/nonfaces1000.npy').astype(np.float32)  # (1000, 31, 25)

# Learn eigenfaces from the first 500 faces (shuffled with a fixed seed)
order = np.random.default_rng(0).permutation(1000)
train, test = order[:500], order[500:]
train_vectors = faces[train].reshape(500, -1)
train_vectors = (train_vectors - train_vectors.mean(axis=1, keepdims=True)) / train_vectors.std(axis=1, keepdims=True)
mean_face = np.mean(train_vectors, axis=0).reshape(31, 25)
face_eigenvectors = computePCA(train_vectors, mean_face.flatten(), n_components=100)

In [ ]:
def pca_score(window, mean_face, eigenvectors, k):
    """Reconstruction error of a window: low means face-like."""
    # Resize the window to the size of the mean face, then flatten it
    window = cv.resize(window, (mean_face.shape[1], mean_face.shape[0]),
                       interpolation=cv.INTER_LINEAR)
    window = window.astype(np.float32).flatten()

    # Normalize the window (a flat window has std 0: leave it at zero)
    window_std = np.std(window)
    window = (window - np.mean(window)) / (window_std if window_std > 0 else 1)

    top_eigenvectors = eigenvectors[:, :k]
    mean_vector = mean_face.flatten()
    projection = pca_project(window, top_eigenvectors, mean_vector)
    reconstructed = pca_reconstruct(projection, top_eigenvectors, mean_vector)

    # Sum of squared differences between the window and its reconstruction
    diff = reconstructed - window
    return np.sum(diff * diff)

k = 10
face_scores = np.array([pca_score(w, mean_face, face_eigenvectors, k) for w in faces[test]])
nonface_scores = np.array([pca_score(w, mean_face, face_eigenvectors, k) for w in nonfaces[test]])

plt.figure(figsize=(8, 4))
bins = np.linspace(0, 1500, 61)
plt.hist(face_scores, bins, alpha=0.6, label='faces')
plt.hist(nonface_scores, bins, alpha=0.6, label='non-faces')
plt.xlabel(f'reconstruction error with k = {k} eigenfaces'), plt.ylabel('windows'), plt.legend()
plt.show()
print('Median error: faces', np.median(face_scores).round(1), ' non-faces', np.median(nonface_scores).round(1))

The two histograms barely overlap: almost every face reconstructs better than almost every non-face. To put one number on that, use the *AUC*: the probability that a randomly chosen face scores better (lower) than a randomly chosen non-face. 1.0 is perfect separation and 0.5 is a coin flip.

Every $k$ defines a different model of what a face is. Watch what happens as $k$ grows.

In [ ]:
from sklearn.metrics import roc_auc_score

labels = np.r_[np.zeros(500), np.ones(500)]     # 1 = non-face, which should score higher
for k in (1, 5, 10, 20, 40, 80):
    f = [pca_score(w, mean_face, face_eigenvectors, k) for w in faces[test]]
    nf = [pca_score(w, mean_face, face_eigenvectors, k) for w in nonfaces[test]]
    print(f'k = {k:2d}: AUC {roc_auc_score(labels, np.r_[f, nf]):.3f}')

Separation is best around $k = 10$ and gets *worse* with more eigenfaces. With many eigenfaces the model can reconstruct almost anything, non-faces included, so the error stops telling them apart. A good face model is one that is good at faces and bad at everything else.

For comparison, here is the template matching score from the template matching lecture: the normalized correlation of each window with the mean face. On these small, well-aligned windows it separates the two sets about as well. Where the eigenface score earns its keep is variety: there is one perfect face for a template (the mean face), but a whole space of perfect faces for PCA.

In [ ]:
def correlation_with_mean(window):
    w = window.flatten() - window.mean()
    m = mean_face.flatten() - mean_face.mean()
    return np.dot(w, m) / (np.linalg.norm(w) * np.linalg.norm(m) + 1e-9)

ncc = [-correlation_with_mean(w) for w in faces[test]] + [-correlation_with_mean(w) for w in nonfaces[test]]
print(f'Normalized correlation with the mean face: AUC {roc_auc_score(labels, ncc):.3f}')

## Try It Yourself

1. Reconstruct a different face, for example number 35 (thick glasses) or number 263 (looking to the side), with 10 and with 100 eigenfaces. What does the 10-number version lose?
2. Load `data/palm-tree.png` (the palm tree from the lecture) or any photo, resize it to $60 \times 50$ with `cv.resize(img, (50, 60))`, normalize it like the faces, and reconstruct it with 10 and with 100 eigenfaces. What do you get, and how does its error compare with a face's?
3. How many eigenfaces keep 90% of the variance? 99%?
4. In the faces vs. non-faces section, which non-faces get the *lowest* reconstruction error? Show the five worst offenders and say what they have in common.
5. The eigenfaces were learned from one particular set of 337 people. If someone's face looked unlike any of them, what would happen to their reconstruction error, and what would that mean for a face detector built on it?